# 01_Data_Loading_Cleaning

Rossmann Sales Forecasting — data loading, quality checks, feature engineering, store merge, and creation of the working dataset. Existing executed outputs are preserved from the completed project notebook.


## Data Loading & Structure


In [ ]:
import pandas as pd

train = pd.read_csv('/content/train.csv')
store = pd.read_csv('/content/store.csv')

print("Train dataset shape:", train.shape)
print("Store dataset shape:", store.shape)

Train dataset shape: (1017209, 9)
Store dataset shape: (1115, 10)


/tmp/ipykernel_4891/1444318771.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('/content/train.csv')


In [1]:
# Python Logging Setup
import logging

logger = logging.getLogger("rossmann_project")
logger.setLevel(logging.INFO)

# Avoid adding duplicate handlers if the cell is run multiple times
if not logger.handlers:
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s - %(levelname)s - %(message)s"
    )
    console_handler.setFormatter(formatter)

    logger.addHandler(console_handler)

logger.info("Rossmann Sales Forecasting project logging initialized successfully.")
print("LOGGING TEST: Rossmann Sales Forecasting project logging initialized successfully.")

2026-09-11 11:28:41,132 - INFO - Rossmann Sales Forecasting project logging initialized successfully.


LOGGING TEST: Rossmann Sales Forecasting project logging initialized successfully.


In [ ]:
print("TRAIN COLUMNS:")
print(train.columns.tolist())

print("\nSTORE COLUMNS:")
print(store.columns.tolist())

print("\nTRAIN DATA TYPES:")
print(train.dtypes)

print("\nSTORE DATA TYPES:")
print(store.dtypes)

TRAIN COLUMNS:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']

STORE COLUMNS:
['Store', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']

TRAIN DATA TYPES:
Store             int64
DayOfWeek         int64
Date             object
Sales             int64
Customers         int64
Open              int64
Promo             int64
StateHoliday     object
SchoolHoliday     int64
dtype: object

STORE DATA TYPES:
Store                          int64
StoreType                     object
Assortment                    object
CompetitionDistance          float64
CompetitionOpenSinceMonth    float64
CompetitionOpenSinceYear     float64
Promo2                         int64
Promo2SinceWeek              float64
Promo2SinceYear              float64
PromoInterval                 object
dtype: object


In [ ]:
train['Date'] = pd.to_datetime(train['Date'])

print(train['Date'].dtype)
print(train['Date'].min())
print(train['Date'].max())

datetime64[ns]
2013-01-01 00:00:00
2015-07-31 00:00:00


## Data Quality & Missing Values


In [ ]:
print("TRAIN MISSING VALUES:")
print(train.isnull().sum())

print("\nSTORE MISSING VALUES:")
print(store.isnull().sum())

TRAIN MISSING VALUES:
Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

STORE MISSING VALUES:
Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64


In [ ]:
print("Promo2 distribution:")
print(store['Promo2'].value_counts())

print("\nMissing Promo2SinceWeek by Promo2:")
print(store.groupby('Promo2')['Promo2SinceWeek'].apply(lambda x: x.isnull().sum()))

print("\nMissing Promo2SinceYear by Promo2:")
print(store.groupby('Promo2')['Promo2SinceYear'].apply(lambda x: x.isnull().sum()))

print("\nMissing PromoInterval by Promo2:")
print(store.groupby('Promo2')['PromoInterval'].apply(lambda x: x.isnull().sum()))

Promo2 distribution:
Promo2
1    571
0    544
Name: count, dtype: int64

Missing Promo2SinceWeek by Promo2:
Promo2
0    544
1      0
Name: Promo2SinceWeek, dtype: int64

Missing Promo2SinceYear by Promo2:
Promo2
0    544
1      0
Name: Promo2SinceYear, dtype: int64

Missing PromoInterval by Promo2:
Promo2
0    544
1      0
Name: PromoInterval, dtype: int64


In [ ]:
print(store[store['CompetitionDistance'].isnull()])

     Store StoreType Assortment  CompetitionDistance  \
290    291         d          a                  NaN   
621    622         a          c                  NaN   
878    879         d          a                  NaN   

     CompetitionOpenSinceMonth  CompetitionOpenSinceYear  Promo2  \
290                        NaN                       NaN       0   
621                        NaN                       NaN       0   
878                        NaN                       NaN       1   

     Promo2SinceWeek  Promo2SinceYear    PromoInterval  
290              NaN              NaN              NaN  
621              NaN              NaN              NaN  
878              5.0           2013.0  Feb,May,Aug,Nov  


In [ ]:
print("Duplicate rows in train:", train.duplicated().sum())
print("Duplicate rows in store:", store.duplicated().sum())

Duplicate rows in train: 0
Duplicate rows in store: 0


## Sales Quality Checks


In [ ]:
print("Sales statistics:")
print(train['Sales'].describe())

print("\nNumber of zero-sales records:")
print((train['Sales'] == 0).sum())

print("\nNumber of negative-sales records:")
print((train['Sales'] < 0).sum())

Sales statistics:
count    1.017209e+06
mean     5.773819e+03
std      3.849926e+03
min      0.000000e+00
25%      3.727000e+03
50%      5.744000e+03
75%      7.856000e+03
max      4.155100e+04
Name: Sales, dtype: float64

Number of zero-sales records:
172871

Number of negative-sales records:
0


In [ ]:
print("Sales = 0 when store is closed:")
print(train.loc[train['Open'] == 0, 'Sales'].eq(0).all())

print("\nNumber of closed-store records:")
print((train['Open'] == 0).sum())

print("\nNumber of closed-store records with non-zero sales:")
print(((train['Open'] == 0) & (train['Sales'] != 0)).sum())

Sales = 0 when store is closed:
True

Number of closed-store records:
172817

Number of closed-store records with non-zero sales:
0


## Calendar Feature Engineering


In [ ]:
train['Year'] = train['Date'].dt.year

print(train[['Date', 'Year']].head())

        Date  Year
0 2015-07-31  2015
1 2015-07-31  2015
2 2015-07-31  2015
3 2015-07-31  2015
4 2015-07-31  2015


In [ ]:
train['Month'] = train['Date'].dt.month
train['Day'] = train['Date'].dt.day
train['WeekOfYear'] = train['Date'].dt.isocalendar().week.astype(int)

print(train[['Date', 'Year', 'Month', 'Day', 'WeekOfYear']].head())

        Date  Year  Month  Day  WeekOfYear
0 2015-07-31  2015      7   31          31
1 2015-07-31  2015      7   31          31
2 2015-07-31  2015      7   31          31
3 2015-07-31  2015      7   31          31
4 2015-07-31  2015      7   31          31


In [ ]:
train['IsWeekend'] = train['DayOfWeek'].isin([6, 7]).astype(int)

print(train[['DayOfWeek', 'IsWeekend']].drop_duplicates().sort_values('DayOfWeek'))

      DayOfWeek  IsWeekend
4460          1          0
3345          2          0
2230          3          0
1115          4          0
0             5          0
6690          6          1
5575          7          1


In [ ]:
print("Current train columns:")
print(train.columns.tolist())

print("\nDataset shape:")
print(train.shape)

Current train columns:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend']

Dataset shape:
(1017209, 14)


In [ ]:
train['YearMonth'] = train['Date'].dt.to_period('M').astype(str)

print(train[['Date', 'YearMonth']].head())

        Date YearMonth
0 2015-07-31   2015-07
1 2015-07-31   2015-07
2 2015-07-31   2015-07
3 2015-07-31   2015-07
4 2015-07-31   2015-07


## Store Merge & Data Validation


In [ ]:
merged = train.merge(
    store,
    on='Store',
    how='left'
)

print("Merged dataset shape:", merged.shape)
print("\nMerged columns:")
print(merged.columns.tolist())

Merged dataset shape: (1017209, 24)

Merged columns:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend', 'YearMonth', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']


In [ ]:
print("Missing StoreType after merge:")
print(merged['StoreType'].isnull().sum())

print("\nMissing Assortment after merge:")
print(merged['Assortment'].isnull().sum())

print("\nNumber of unique stores in train:", train['Store'].nunique())
print("Number of unique stores in store:", store['Store'].nunique())
print("Number of unique stores in merged:", merged['Store'].nunique())

Missing StoreType after merge:
0

Missing Assortment after merge:
0

Number of unique stores in train: 1115
Number of unique stores in store: 1115
Number of unique stores in merged: 1115


In [ ]:
print("StoreType:")
print(merged['StoreType'].value_counts())

print("\nAssortment:")
print(merged['Assortment'].value_counts())

print("\nStateHoliday:")
print(merged['StateHoliday'].value_counts())

print("\nPromoInterval:")
print(merged['PromoInterval'].value_counts(dropna=False))

StoreType:
StoreType
a    551627
d    312912
c    136840
b     15830
Name: count, dtype: int64

Assortment:
Assortment
a    537445
c    471470
b      8294
Name: count, dtype: int64

StateHoliday:
StateHoliday
0    855087
0    131072
a     20260
b      6690
c      4100
Name: count, dtype: int64

PromoInterval:
PromoInterval
NaN                 508031
Jan,Apr,Jul,Oct     293122
Feb,May,Aug,Nov     118596
Mar,Jun,Sept,Dec     97460
Name: count, dtype: int64


In [ ]:
print(merged['StateHoliday'].map(type).value_counts())

StateHoliday
<class 'str'>    886137
<class 'int'>    131072
Name: count, dtype: int64


In [ ]:
merged['StateHoliday'] = merged['StateHoliday'].astype(str)

print(merged['StateHoliday'].map(type).value_counts())
print("\nStateHoliday values:")
print(merged['StateHoliday'].value_counts())

StateHoliday
<class 'str'>    1017209
Name: count, dtype: int64

StateHoliday values:
StateHoliday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64


In [ ]:
open_data = merged[merged['Open'] == 1]

print("Open-store records:", len(open_data))
print("Zero-sales records while open:", (open_data['Sales'] == 0).sum())
print("Average sales while open:", open_data['Sales'].mean())
print("Maximum sales while open:", open_data['Sales'].max())

Open-store records: 844392
Zero-sales records while open: 54
Average sales while open: 6955.514290755952
Maximum sales while open: 41551


In [ ]:
# Create a working copy for machine learning
df = merged.copy()

print("Final working dataset shape:", df.shape)
print("Total missing values:", df.isnull().sum().sum())

Final working dataset shape: (1017209, 24)
Total missing values: 2173431


## Project Objective

The objective of this stage is to load the Rossmann Train and Store datasets, validate their structure and data quality, perform safe cleaning and calendar feature engineering, and create the merged analytical dataset used by later EDA and modelling stages.

### Cleaning decisions
- Duplicate records are checked rather than removed without evidence.
- Negative Sales values are checked and none are present.
- Records with `Open = 0` correctly have `Sales = 0`; these records are retained because they are valid business observations.
- `StateHoliday` is standardized to a consistent string representation.
- Structural missing values in Promo2-related fields are retained because they represent stores where Promo2 is not active.
- Missing `CompetitionDistance` and other store attributes are retained for downstream pipeline-based imputation rather than dropping large portions of data.

### Final dataset hand-off
The merged dataset contains 1,017,209 records and 24 columns. `Sales` is the primary target for the initial forecasting model. Calendar features and store/business attributes are carried forward to the EDA and ML notebooks.

# Data Cleaning Decisions and Modelling Handoff

### Cleaning decisions
- Duplicate records were checked and no duplicate rows were identified.
- Sales quality was checked: there were no negative Sales values.
- Rows where `Open = 0` correctly correspond to zero Sales in the training data.
- `StateHoliday` was standardized to a consistent string representation.
- Structural missing values associated with `Promo2` were retained because they represent stores where the feature is not applicable.
- Missing `CompetitionDistance` values were retained for later pipeline-based imputation rather than dropping observations.

### Modelling handoff
The merged modelling dataset contains 24 columns. `Sales` is the target for the initial sales model. `Customers` is not used as an input to the initial Sales model because future customer counts would not be available at prediction time.

The cleaned and merged dataset is therefore passed forward to EDA and modelling without unnecessary row deletion.
